# CKC Register

This notebook is the **single source of truth** for navigation.

Run the generator cell to:
- scan `data_science/`, `backend/`, `frontend/`, `devops/`, `cybersecurity/`
- list all notebooks by section/subsection/topic
- (optionally) create missing `00_overview.ipynb` notebooks for leaf folders
- update the auto-generated index block in this notebook and in `README.md`


## Index (auto-generated)

<!-- CKC_INDEX_START -->

### Data Science
- **Machine Learning**
  - **Supervised Learning**
    - **Gaussian Processes**
      - [Gaussian Processes (GP) — Regression & Probabilistic Classification](data_science/machine_learning/supervised_learning/gaussian_processes/00_overview.ipynb)
    - **Generalized Linear Models**
      - [Generalized Linear Models (GLMs)](data_science/machine_learning/supervised_learning/generalized_linear_models/00_overview.ipynb)
    - **K Nearest Neighbors**
      - [K-Nearest Neighbors (KNN) — Classification & Regression (From Scratch)](data_science/machine_learning/supervised_learning/k_nearest_neighbors/00_overview.ipynb)
    - **Linear And Quadratic Discriminant Analysis**
      - [Linear and Quadratic Discriminant Analysis (LDA / QDA)](data_science/machine_learning/supervised_learning/linear_and_quadratic_discriminant_analysis/00_overview.ipynb)
    - **Linear Regression And Co**
      - [Linear Regression & Friends (OLS, Ridge, Lasso, Elastic Net)](data_science/machine_learning/supervised_learning/linear_regression_and_co/00_overview.ipynb)
    - **Naive Bayes**
      - [Naive Bayes (Gaussian, Multinomial, Complement, Bernoulli, Categorical) + Out-of-core](data_science/machine_learning/supervised_learning/naive_bayes/00_overview.ipynb)
    - **Support Vector Machines**
      - [Support Vector Machines (SVC + SVR)](data_science/machine_learning/supervised_learning/support_vector_machines/00_overview.ipynb)
    - **Tree Based Algorithms**
      - [Tree-Based Algorithms (Decision Trees, Random Forests, Gradient Boosting)](data_science/machine_learning/supervised_learning/tree_based_algorithms/00_overview.ipynb)

### Backend

### Frontend

### DevOps

### Cybersecurity


<!-- CKC_INDEX_END -->


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
except Exception:  # pragma: no cover
    Markdown = None
    display = None


# --- config ---
ROOT = Path(".")
REGISTER_NOTEBOOK = Path("CKC_REGISTER.ipynb")
README_PATH = Path("README.md")

SECTIONS = [
    "data_science",
    "backend",
    "frontend",
    "devops",
    "cybersecurity",
]

SKIP_DIR_NAMES = {
    ".git",
    ".ipynb_checkpoints",
    "__pycache__",
    "node_modules",
    ".venv",
    "venv",
}

INDEX_START = "<!-- CKC_INDEX_START -->"
INDEX_END = "<!-- CKC_INDEX_END -->"

AUTO_CREATE_MISSING_LEAF_OVERVIEWS = True

SPECIAL_DIR_DISPLAY = {
    "data_science": "Data Science",
    "devops": "DevOps",
    "iac": "IaC",
    "cicd": "CI/CD",
    "nextjs": "Next.js",
    "fastapi": "FastAPI",
}


def _is_skipped_dir(path: Path) -> bool:
    name = path.name
    if name in SKIP_DIR_NAMES:
        return True
    if name.startswith("."):
        return True
    return False


def _humanize_dir_name(name: str) -> str:
    if name in SPECIAL_DIR_DISPLAY:
        return SPECIAL_DIR_DISPLAY[name]
    name = name.replace("_", " ").strip()
    return " ".join(w[:1].upper() + w[1:] if w else w for w in name.split())


def _notebook_title(nb_path: Path) -> str:
    try:
        data = json.loads(nb_path.read_text(encoding="utf-8"))
    except Exception:
        return nb_path.stem

    for cell in data.get("cells", []):
        if cell.get("cell_type") != "markdown":
            continue

        src = cell.get("source", "")
        text = "".join(src) if isinstance(src, list) else str(src)
        for line in text.splitlines():
            line = line.strip()
            if line.startswith("#"):
                return line.lstrip("#").strip() or nb_path.stem

    return nb_path.stem


def _leaf_dirs(section_dir: Path) -> list[Path]:
    leaves: list[Path] = []
    for d in sorted(section_dir.rglob("*")):
        if not d.is_dir():
            continue

        rel_parts = d.relative_to(ROOT).parts
        if any((p in SKIP_DIR_NAMES) or p.startswith(".") for p in rel_parts):
            continue

        subdirs = [p for p in d.iterdir() if p.is_dir() and not _is_skipped_dir(p)]
        if not subdirs:
            leaves.append(d)

    return leaves


def _make_overview_notebook(title: str) -> dict:
    return {
        "cells": [
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": [
                    f"# {title}\n",
                    "\n",
                    "## Topics\n",
                    "- (add notebooks in this folder)\n",
                    "\n",
                    "## Notes\n",
                    "- (key takeaways / gotchas / references)\n",
                ],
            }
        ],
        "metadata": {
            "kernelspec": {
                "display_name": "Python 3",
                "language": "python",
                "name": "python3",
            },
            "language_info": {
                "name": "python",
                "pygments_lexer": "ipython3",
            },
        },
        "nbformat": 4,
        "nbformat_minor": 5,
    }


def _ensure_leaf_overviews() -> list[Path]:
    missing: list[Path] = []

    for section in SECTIONS:
        section_dir = ROOT / section
        if not section_dir.exists():
            continue

        for leaf in _leaf_dirs(section_dir):
            notebooks = [p for p in leaf.glob("*.ipynb") if p.name != REGISTER_NOTEBOOK.name]
            if notebooks:
                continue

            missing.append(leaf)
            if not AUTO_CREATE_MISSING_LEAF_OVERVIEWS:
                continue

            title = _humanize_dir_name(leaf.name)
            out_path = leaf / "00_overview.ipynb"
            nb = _make_overview_notebook(title)
            out_path.write_text(json.dumps(nb, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    return missing


def _dir_index_lines(dir_path: Path, depth: int) -> list[str]:
    lines: list[str] = []

    # notebooks in this directory
    for nb in sorted(dir_path.glob("*.ipynb"), key=lambda p: p.name):
        if nb.name == REGISTER_NOTEBOOK.name:
            continue
        rel = nb.relative_to(ROOT).as_posix()
        title = _notebook_title(nb)
        lines.append(f"{'  ' * depth}- [{title}]({rel})")

    # subdirectories
    subdirs = [p for p in dir_path.iterdir() if p.is_dir() and not _is_skipped_dir(p)]
    for sub in sorted(subdirs, key=lambda p: p.name):
        lines.append(f"{'  ' * depth}- **{_humanize_dir_name(sub.name)}**")
        child = _dir_index_lines(sub, depth + 1)
        lines.extend(child if child else [f"{'  ' * (depth + 1)}- (no notebooks yet)"])

    return lines


def build_index_markdown() -> str:
    parts: list[str] = []
    for section in SECTIONS:
        section_dir = ROOT / section
        if not section_dir.exists():
            continue

        parts.append(f"### {_humanize_dir_name(section)}")
        parts.extend(_dir_index_lines(section_dir, depth=0))
        parts.append("")

    return "\n".join(parts).strip() + "\n"


def _replace_marked_block(text: str, new_block: str) -> str:
    if INDEX_START not in text or INDEX_END not in text:
        return text.rstrip() + f"\n\n## Index (auto-generated)\n\n{INDEX_START}\n\n{new_block}\n\n{INDEX_END}\n"

    pre, rest = text.split(INDEX_START, 1)
    _, post = rest.split(INDEX_END, 1)
    return pre + INDEX_START + "\n\n" + new_block + "\n\n" + INDEX_END + post


def update_register_notebook(index_md: str) -> None:
    data = json.loads(REGISTER_NOTEBOOK.read_text(encoding="utf-8"))

    for cell in data.get("cells", []):
        if cell.get("cell_type") != "markdown":
            continue

        src = cell.get("source", "")
        text = "".join(src) if isinstance(src, list) else str(src)
        if INDEX_START in text and INDEX_END in text:
            updated = _replace_marked_block(text, index_md)
            cell["source"] = updated.splitlines(keepends=True)
            break
    else:
        data.setdefault("cells", []).append(
            {
                "cell_type": "markdown",
                "metadata": {},
                "source": _replace_marked_block("", index_md).splitlines(keepends=True),
            }
        )

    REGISTER_NOTEBOOK.write_text(json.dumps(data, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def update_readme(index_md: str) -> None:
    if not README_PATH.exists():
        return

    text = README_PATH.read_text(encoding="utf-8")
    README_PATH.write_text(_replace_marked_block(text, index_md), encoding="utf-8")


# --- run ---
missing_leaf_dirs = _ensure_leaf_overviews()
index_md = build_index_markdown()
update_register_notebook(index_md)
update_readme(index_md)

if missing_leaf_dirs:
    print("Leaf folders missing notebooks (before auto-create):")
    for p in missing_leaf_dirs:
        print(" -", p)
else:
    print("All leaf folders contain at least one notebook.")

if Markdown is not None and display is not None:
    display(Markdown(index_md))
else:
    print(index_md)
